In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

In [ ]:
%pip install -U pyautogen

In [ ]:
%pip install -U autogen-ext[openai]

In [ ]:
load_dotenv()

openai_endpoint = os.getenv("ENDPOINT")
model_deployment = os.getenv("MODEL_DEPLOYMENT")

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [31]:
model_client = OpenAIChatCompletionClient(
    model=model_deployment,
    base_url=openai_endpoint,
    api_key=token_provider(),
)

In [ ]:
agent = AssistantAgent(
    name="chatbot",
    model_client=model_client,
    system_message="You are a helpful assistant."
)


In [ ]:
response = await agent.on_messages(
    [TextMessage(content="Tell me a joke.", source="user")],
    cancellation_token=None,
)

print(response.chat_message.content)

In [ ]:
response = await agent.on_messages(
    [TextMessage(content="Repeat joke", source="user")],
    cancellation_token=None,
)

print(response.chat_message.content)

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination

# Cathy
cathy = AssistantAgent(
    name="cathy",
    model_client=model_client,
    system_message="Your name is Cathy and you are a stand-up comedian."
)

# Joe
joe = AssistantAgent(
    name="joe",
    model_client=model_client,
    system_message=(
        "Your name is Joe and you are a stand-up comedian. "
        "Start the next joke from the punchline of the previous joke."
    )
)

# Stop after 2 turns
termination = MaxMessageTermination(2)

# Team conversation
team = RoundRobinGroupChat(
    participants=[joe, cathy],
    termination_condition=termination,
)

# Start chat
result = await team.run(
    task="I'm Joe. Cathy, let's keep the jokes rolling."
)

print(result)

In [ ]:
for msg in result.messages:
    print(f"\n{msg.source}:")
    print(msg.content)

In [ ]:
from autogen_agentchat.agents import AssistantAgent

personal_info_agent = AssistantAgent(
    name="personal_info_agent",
    model_client=model_client,
    system_message="""
    You are a helpful customer onboarding agent.
    Gather the customer's name and location.
    Do not ask for other information.
    Return TERMINATE when finished.
    """
)

topic_preference_agent = AssistantAgent(
    name="topic_preference_agent",
    model_client=model_client,
    system_message="""
    You are a helpful customer onboarding agent.
    Gather the customer's preferences on news topics.
    Do not ask for other information.
    Return TERMINATE when finished.
    """
)

engagement_agent = AssistantAgent(
    name="engagement_agent",
    model_client=model_client,
    system_message="""
    You are a helpful customer service agent.

    Based on the user's personal information and topic preferences,
    suggest interesting articles, stories, fun facts, or jokes.

    Return TERMINATE when finished.
    """
)

In [ ]:
from autogen_agentchat.messages import TextMessage

# Step 1 - Personal Information
response1 = await personal_info_agent.on_messages(
    [
        TextMessage(
            content="""
            Customer says:
            My name is Jeevita and I live in Chennai.
            """,
            source="user"
        )
    ],
    cancellation_token=None
)

personal_info = response1.chat_message.content

print("=== Personal Information ===")
print(personal_info)

In [ ]:
response2 = await topic_preference_agent.on_messages(
    [
        TextMessage(
            content="""
            I enjoy reading about AI, cloud computing,
            software engineering, and Java development.
            """,
            source="user"
        )
    ],
    cancellation_token=None
)

preferences = response2.chat_message.content

print("=== Preferences ===")
print(preferences)

In [ ]:
response3 = await engagement_agent.on_messages(
    [
        TextMessage(
            content=f"""
            Customer Information:

            {personal_info}

            Topic Preferences:

            {preferences}

            Please suggest something interesting.
            """,
            source="user"
        )
    ],
    cancellation_token=None
)

print("=== Final Engagement ===")
print(response3.chat_message.content)

In [27]:
from autogen_agentchat.messages import TextMessage

results = []

# Chat 1 - Personal Information
response1 = await personal_info_agent.on_messages(
    [
        TextMessage(
            content="My name is Jeevita and I live in Chennai.",
            source="user"
        )
    ],
    cancellation_token=None,
)

results.append({
    "name": "Personal Information",
    "summary": response1.chat_message.content
})

# Chat 2 - Preferences
response2 = await topic_preference_agent.on_messages(
    [
        TextMessage(
            content="I like AI, Azure, Java and Cloud Computing.",
            source="user"
        )
    ],
    cancellation_token=None,
)

results.append({
    "name": "Topic Preferences",
    "summary": response2.chat_message.content
})

# Chat 3 - Engagement
response3 = await engagement_agent.on_messages(
    [
        TextMessage(
            content=f"""
            Customer Information:
            {response1.chat_message.content}

            Preferences:
            {response2.chat_message.content}
            """,
            source="user"
        )
    ],
    cancellation_token=None,
)

results.append({
    "name": "Customer Engagement",
    "summary": response3.chat_message.content
})

In [28]:
for result in results:
    print("=" * 50)
    print(result["name"])
    print("=" * 50)
    print(result["summary"])
    print()

Personal Information
Thanks, Jeevita — I’ve noted your name and that you’re in Chennai.

TERMINATE

Topic Preferences
Thanks — a few quick focused questions (please reply with the numbers you want to answer):

1) AI — which subareas? (generative AI, ML research, NLP, computer vision, MLOps, ethics/regulation, AI product launches)

2) Azure/cloud — which Azure topics or services? (AKS/Kubernetes, Azure Functions/serverless, Azure AI services, architecture, cost optimization, security, infra-as-code/ARM/Bicep, DevOps/GitHub Actions)

3) Java — which focus areas? (Spring/Spring Boot, Jakarta EE, JVM performance/tuning, language features, frameworks, Kotlin interop, tooling/IDEs)

4) Types of content you prefer — pick any: product releases, tutorials/how‑tos with code, in‑depth technical analyses, research papers, opinion pieces, conference coverage, security advisories, company/business/funding news

5) Technical depth — prefer: beginner, intermediate, advanced, or a mix?

6) Any other te

In [30]:
for response in [response1, response2, response3]:
    usage = response.chat_message.models_usage
    print(usage)

RequestUsage(prompt_tokens=106, completion_tokens=221)
RequestUsage(prompt_tokens=359, completion_tokens=991)
RequestUsage(prompt_tokens=1238, completion_tokens=1135)
